# 01B — Common canonical adapter

**Outcome:** convert either valid sector pack into the same Tier-0
episode-aware `SPEC-CORE`, with `SPLITS` and physically separate
`SPEC-EVAL`.

This notebook contains no ONT, splitter, well, valve or native metric logic.


## 1. Setup and sector switch

Run the relevant 01A notebook first. Change only `SECTOR` to switch sources.
`AS_OF_TS` is optional; when set, only observations available at or before
that timestamp may enter the canonical run.


In [ ]:
import os
import shutil
import sys
import tempfile
from pathlib import Path

import pandas as pd
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "milestone1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from milestone1_core import (
    CORE_SCHEMAS,
    CORE_VERSION,
    PACK_ENTITY_SCHEMA,
    PACK_EPISODE_SCHEMA,
    PACK_METRIC_SCHEMA,
    PACK_OBSERVATION_SCHEMA,
    check_core,
    core_fingerprint,
    save_pack,
    build_canonical,
    read_json,
    load_pack,
)

# Change only this value when moving between sectors.
SECTOR = os.getenv("ADAPTER_SECTOR", "telecom")
BUILD_CANONICAL = os.getenv("BUILD_CANONICAL", "1") == "1"
AS_OF_TS = os.getenv("CANONICAL_AS_OF_TS") or None

PACK_RUN_IDS = {
    "telecom": "telecom_pack_v0_6_1",
    "petrobras_3w": "real_wells_expanded_v0_6",
}
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_9_topology_run1",
    "petrobras_3w": "petrobras_3w_core_v0_9_run1",
}
if SECTOR not in PACK_RUN_IDS:
    raise ValueError(f"Choose one of {list(PACK_RUN_IDS)}")

PACK_RUN_ID = os.getenv("ADAPTER_PACK_RUN_ID", PACK_RUN_IDS[SECTOR])
CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
PACK_ROOT = DRIVE_ROOT / "outputs" / "packs" / SECTOR / PACK_RUN_ID
RUN_ROOT = DRIVE_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / CANONICAL_RUN_ID

display(pd.Series({
    "sector": SECTOR,
    "pack_root": str(PACK_ROOT),
    "canonical_root": str(RUN_ROOT),
    "build": BUILD_CANONICAL,
    "as_of_ts": AS_OF_TS or "all observations",
}, name="value").to_frame())


## 2. Inspect and materialise the frozen contract

`SPEC-CORE` contains only telemetry, authored metric semantics,
sector-supplied quality codes, observation-derived entity and episode
bounds, plus periodic collection gaps. Labels
stay in the physically separate `SPEC-EVAL` directory.


In [ ]:
display(pd.DataFrame([
    {"table": name, "columns": ", ".join(columns)}
    for name, columns in CORE_SCHEMAS.items()
]))

pack_manifest = load_pack(PACK_ROOT)
display(pd.Series(pack_manifest, name="value").to_frame())

if BUILD_CANONICAL:
    run_manifest = build_canonical(
        PACK_ROOT,
        RUN_ROOT,
        include_evaluation=True,
        as_of_ts=AS_OF_TS,
    )
else:
    run_manifest = read_json(RUN_ROOT / "run_manifest.json")

core_audit = check_core(RUN_ROOT / "SPEC-CORE")
display(pd.Series(core_audit, name="value").to_frame())


## 3. Runtime truth isolation

This is deployment evidence, not the primary leakage proof. The primary
proof remains each 01A original-versus-redacted translator test.

Here the same pack is materialised with and without evaluation mounted.
`SPEC-CORE` fingerprints must match. The negative
control deliberately reads `SPEC-EVAL` and must fail when it is absent.


In [ ]:
def deliberately_leaky_scorer(run_root):
    path = Path(run_root) / "SPEC-EVAL"
    if not path.is_dir():
        raise FileNotFoundError("SPEC-EVAL is not mounted")
    return sorted(file.name for file in path.glob("*.parquet"))


def run_runtime_isolation_test():
    with tempfile.TemporaryDirectory() as temporary:
        temporary = Path(temporary)
        mounted = temporary / "mounted"
        unmounted = temporary / "unmounted"
        build_canonical(PACK_ROOT, mounted, include_evaluation=True, as_of_ts=AS_OF_TS)
        build_canonical(PACK_ROOT, unmounted, include_evaluation=False, as_of_ts=AS_OF_TS)

        assert core_fingerprint(mounted / "SPEC-CORE") == core_fingerprint(unmounted / "SPEC-CORE")
        deliberately_leaky_scorer(mounted)
        try:
            deliberately_leaky_scorer(unmounted)
        except FileNotFoundError:
            pass
        else:
            raise AssertionError("Negative control unexpectedly read unmounted truth")

if pack_manifest["evaluation_tables"]:
    run_runtime_isolation_test()
    isolation_status = "pass"
    print("PASS — SPEC-CORE and runtime output are truth-invariant")
    print("PASS — the negative control fails without SPEC-EVAL")
else:
    isolation_status = "not_run_no_evaluation"
    print("NOT RUN — runtime truth isolation requires a labelled development pack")


## 4. Metric-level cadence contract

This generic fixture places a one-second and a five-second metric in the
same episode. The slow metric deliberately misses its expected five-second
observation. Unscheduled seconds must not become invalid slow-metric rows,
and the missing scheduled observation must become one collection gap.


In [ ]:
def run_mixed_cadence_contract_test():
    base = pd.Timestamp("2025-01-01 00:00:00", tz="UTC")
    catalogue = pd.DataFrame([
        ("fast_signal", "test_asset", "gauge", "unit", "periodic", 1),
        ("slow_signal", "test_asset", "gauge", "unit", "periodic", 5),
    ], columns=PACK_METRIC_SCHEMA)
    fast = pd.DataFrame({
        "event_ts": [base + pd.Timedelta(seconds=value) for value in range(16)],
        "entity_id": "asset-1",
        "episode_id": "asset-1::episode-1",
        "metric_id": "fast_signal",
        "value": range(16),
    })
    slow = pd.DataFrame({
        "event_ts": [base, base + pd.Timedelta(seconds=10), base + pd.Timedelta(seconds=15)],
        "entity_id": "asset-1",
        "episode_id": "asset-1::episode-1",
        "metric_id": "slow_signal",
        "value": [20.0, 21.0, None],
    })
    observations = pd.concat([fast, slow], ignore_index=True)
    observations["quality_code"] = "measured"
    observations.loc[observations["value"].isna(), "quality_code"] = "invalid"
    observations = observations[PACK_OBSERVATION_SCHEMA]

    with tempfile.TemporaryDirectory() as temporary:
        temporary = Path(temporary)
        pack = temporary / "pack"
        core = pack / "PACK-CORE"
        (core / "observations").mkdir(parents=True)
        observations.to_parquet(core / "observations" / "part-00000.parquet", index=False)
        catalogue.to_parquet(core / "metric_catalogue.parquet", index=False)
        pd.DataFrame([("asset-1", "test_asset")], columns=PACK_ENTITY_SCHEMA).to_parquet(
            core / "entity_registry.parquet", index=False
        )
        pd.DataFrame([
            ("asset-1::episode-1", "asset-1", "mixed_cadence_fixture")
        ], columns=PACK_EPISODE_SCHEMA).to_parquet(
            core / "observation_episodes.parquet", index=False
        )
        save_pack(
            pack, sector="contract_test", pack_version="metric-presence-v1",
            source_info={"source_id": "mixed-cadence-fixture", "files": []},
        )
        run = temporary / "run"
        build_canonical(pack, run, include_evaluation=False)
        telemetry = pd.concat(
            [pd.read_parquet(path) for path in sorted((run / "SPEC-CORE" / "telemetry").glob("part-*.parquet"))],
            ignore_index=True,
        )
        gaps = pd.read_parquet(run / "SPEC-CORE" / "collection_gaps.parquet")

    slow_rows = telemetry.loc[telemetry["metric_id"].eq("slow_signal")].sort_values("event_ts")
    assert len(slow_rows) == 3
    assert slow_rows.iloc[-1]["quality_code"] == "invalid"
    assert len(gaps) == 1 and gaps.iloc[0]["metric_id"] == "slow_signal"
    assert gaps.iloc[0]["gap_start"] == base + pd.Timedelta(seconds=5)
    assert gaps.iloc[0]["gap_end"] == base + pd.Timedelta(seconds=10)


run_mixed_cadence_contract_test()
print("PASS — metric-level cadence and presence are handled independently")


## 5. Temporal isolation

A small generic pack fixture is created twice: once with future observations
present and once physically truncated. Both are materialised at the same
cutoff. Their canonical content must match, including observation-derived
entity bounds. This catches accidental use of future rows.


In [ ]:
def write_pack_fixture(source_pack, destination, cutoff=None):
    source_pack, destination = Path(source_pack), Path(destination)
    pack_manifest = read_json(source_pack / "pack_manifest.json")
    with tempfile.TemporaryDirectory() as staging_name:
        staging = Path(staging_name) / "pack"
        core = staging / "PACK-CORE"
        observations = core / "observations"
        observations.mkdir(parents=True)

        catalogue = pd.read_parquet(source_pack / "PACK-CORE" / "metric_catalogue.parquet")
        first_part = sorted((source_pack / "PACK-CORE" / "observations").glob("part-*.parquet"))[0]
        sample = pd.read_parquet(first_part).sort_values("event_ts").head(2_000)
        sample["event_ts"] = pd.to_datetime(sample["event_ts"], utc=True)
        if cutoff is not None:
            sample = sample.loc[sample["event_ts"].le(cutoff)]
        if sample.empty:
            raise ValueError("Temporal fixture is empty")
        catalogue = catalogue.loc[
            catalogue["metric_id"].astype(str).isin(sample["metric_id"].astype(str).unique())
        ].copy()
        sample.to_parquet(observations / "part-00000.parquet", index=False)

        registry = pd.read_parquet(source_pack / "PACK-CORE" / "entity_registry.parquet")
        registry = registry.loc[registry["entity_id"].astype(str).isin(sample["entity_id"].astype(str).unique())]
        episodes = pd.read_parquet(source_pack / "PACK-CORE" / "observation_episodes.parquet")
        episodes = episodes.loc[
            episodes["episode_id"].astype(str).isin(sample["episode_id"].astype(str).unique())
        ]
        catalogue.to_parquet(core / "metric_catalogue.parquet", index=False)
        registry[PACK_ENTITY_SCHEMA].to_parquet(core / "entity_registry.parquet", index=False)
        episodes[PACK_EPISODE_SCHEMA].to_parquet(
            core / "observation_episodes.parquet", index=False
        )

        save_pack(
            staging,
            sector=pack_manifest["sector"],
            pack_version="temporal-fixture-v1",
            source_info={"fixture_of": pack_manifest["source"]["source_id"]},
        )
        shutil.copytree(staging, destination)
    return sample


with tempfile.TemporaryDirectory() as temporary:
    temporary = Path(temporary)
    preview = pd.read_parquet(
        sorted((PACK_ROOT / "PACK-CORE" / "observations").glob("part-*.parquet"))[0],
        columns=["event_ts"],
    ).sort_values("event_ts").head(2_000)
    cutoff = pd.to_datetime(preview["event_ts"], utc=True).iloc[len(preview) // 2]

    full_fixture = temporary / "full_pack"
    truncated_fixture = temporary / "truncated_pack"
    write_pack_fixture(PACK_ROOT, full_fixture)
    write_pack_fixture(PACK_ROOT, truncated_fixture, cutoff=cutoff)

    full_run = temporary / "full_as_of"
    truncated_run = temporary / "truncated_as_of"
    build_canonical(full_fixture, full_run, include_evaluation=False, as_of_ts=cutoff)
    build_canonical(truncated_fixture, truncated_run, include_evaluation=False, as_of_ts=cutoff)
    assert core_fingerprint(full_run / "SPEC-CORE") == core_fingerprint(truncated_run / "SPEC-CORE")

print("PASS — observations appended after as_of_ts cannot change earlier canonical content")


## 6. Acceptance checks and output inspection

The final block prints all small tables and manifests. Partitioned telemetry
is summarized rather than printed in full.


In [ ]:
acceptance = {
    "contract_version": CORE_VERSION,
    "sector": SECTOR,
    "pack_valid": True,
    "truth_isolation": isolation_status,
    "negative_control": isolation_status,
    "mixed_cadence_contract": "pass",
    "temporal_isolation": "pass",
    "adapter_sector_branch": False,
    "tier0_tables": list(CORE_SCHEMAS),
}
display(pd.Series(acceptance, name="result").to_frame())

for name in (
    "metric_catalogue", "entity_registry",
    "observation_episodes", "collection_gaps",
):
    path = RUN_ROOT / "SPEC-CORE" / f"{name}.parquet"
    frame = pd.read_parquet(path)
    print(f"\n{name}: {len(frame):,} rows")
    display(frame.head(10))

telemetry_parts = sorted((RUN_ROOT / "SPEC-CORE" / "telemetry").glob("part-*.parquet"))
print(f"\ntelemetry: {len(telemetry_parts):,} parts")
display(pd.read_parquet(telemetry_parts[0]).head(10))

for name in pack_manifest.get("split_tables", []):
    path = RUN_ROOT / "SPLITS" / f"{name}.parquet"
    frame = pd.read_parquet(path)
    print(f"\nSPLITS/{name}: {len(frame):,} rows (not model input)")
    display(frame.head(10))
display(pd.Series(run_manifest, name="value").to_frame())
print("Canonical run:", RUN_ROOT)
print("Next: 02_CANONICAL_EDA.ipynb")
